## CNN Weights Upload to S3 (GroupFuture)

A compact end-to-end example that builds a small CNN in PyTorch and stores all model weights in AWS S3 via LAILA.

### What this notebook does
- Build a simple CNN model in `torch`
- Collect all weight tensors from `model.state_dict()`
- Wrap each tensor as a LAILA entry (`laila.constant`) to get global IDs
- Upload all entries together in one `laila.memorize(...)` call
- Receive a `GroupFuture`, wait for completion, and inspect status
- Fetch one weight back to verify round-trip integrity

### Credentials
This notebook reads AWS credentials from:
`laila.args`


In [ ]:
%load_ext autoreload
%autoreload 2
 

In [ ]:
import laila
import torch
import torch.nn as nn


In [ ]:
from laila.pool import S3Pool

# These argument names are arbitrary and can be changed by the user.
# laila.args is just a container for user-provided arguments.
# Replace these placeholders with your own AWS credentials before running.
laila.args.AWS_BUCKET_NAME = "your-bucket-name"
laila.args.AWS_ACCESS_KEY_ID = "your-access-key-id"
laila.args.AWS_SECRET_ACCESS_KEY = "your-secret-access-key"
laila.args.AWS_REGION = "us-east-1"

s3_pool = S3Pool(
    bucket_name=laila.args.AWS_BUCKET_NAME,
    access_key_id=laila.args.AWS_ACCESS_KEY_ID,
    secret_access_key=laila.args.AWS_SECRET_ACCESS_KEY,
    region_name=laila.args.AWS_REGION,
)


In [ ]:
laila.add_pool(s3_pool, pool_nickname="cnn_weights_s3_pool")


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((8, 8)),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 8 * 8, 64),
            nn.ReLU(),
            nn.Linear(64, 10),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)





In [ ]:
#I DO SOME TRAINING HERE
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN
# TRAIN TRAIN TRAIN

In [ ]:
final_model = SimpleCNN().eval()

In [ ]:
# Wrap every weight tensor as a separate LAILA constant
from typing import Any


weight_entries = {
    name: laila.constant(data=tensor.detach().cpu(), nickname=name)
    for name, tensor in final_model.state_dict().items()
}

print(f"Prepared {len(weight_entries)} weight entries.")
for name, entry in list[tuple[str, Any]](weight_entries.items()):
    print(name, "->", entry.global_id)


In [ ]:
my_model_manifest = laila.constant(
    data = {name:entry.global_id for name, entry in weight_entries.items()},
    nickname = "my_model",
)

In [ ]:
# Upload all weights together to get a GroupFuture
weight_group_future = laila.memorize(
    list(weight_entries.values()),
    pool_nickname="cnn_weights_s3_pool",
)

print("Future type:", type(weight_group_future).__name__)
print("Status before wait:", weight_group_future.status)
weight_group_future.wait()
print("Status after wait:", weight_group_future.status)
print("Underlying futures:", len(weight_group_future))


In [ ]:
manifest_future = laila.memorize(
    my_model_manifest,
    pool_nickname="cnn_weights_s3_pool",
)
manifest_future.wait()


In [ ]:
# Round-trip check on one weight tensor
sample_name = next(iter(weight_entries.keys()))
sample_entry = weight_entries[sample_name]
original_tensor = final_model.state_dict()[sample_name]

remember_future = laila.remember(sample_entry.global_id, pool_nickname="cnn_weights_s3_pool")
remember_future.wait()
recovered_entry = remember_future.result

assert torch.equal(recovered_entry.data, original_tensor)
print("Recovered tensor matches:", sample_name)


In [ ]:
# Optional cleanup: delete uploaded weights from S3
forget_group_future = laila.forget(
    [entry.global_id for entry in weight_entries.values()],
    pool_nickname="cnn_weights_s3_pool",
)
forget_group_future.wait()
print("Cleanup complete.")
